In [1]:
import pandas as pd

text_test_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\data\text_test.csv"
text_test_df = pd.read_csv(text_test_path, keep_default_na=False)

text_test_df.head()

,text,label
0,the runner can leave his base at any time if t...,9
1,well its not an ftp site but i got an number f...,12
2,hi i was reading through the spaceflight handb...,14
3,i was a graduate student in the early s and we...,18
4,freeenergy technology by robert e mcelwaine ph...,0


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)

class NewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]['text']
        label = self.data.iloc[idx]['label']

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        label = torch.tensor(label, dtype=torch.long)
        return input_ids, attention_mask, label

test_dataset = NewsDataset(text_test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [3]:
import torch
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, max_len=256, num_classes=20, num_layers=3, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4, dropout=0.1, batch_first=True)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.size()
        positions = torch.arange(0, T, device=input_ids.device).unsqueeze(0).expand(B, T)
        x = self.embedding(input_ids) + self.pos_embedding(positions)  # [B, T, E]

        # padding mask: True는 마스킹됨
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)

        x = self.norm(x)
        pooled = x.mean(dim=1)  # 평균 풀링
        return self.classifier(pooled)

#######

import torch
import torch.nn as nn
from transformers import DistilBertModel

class BertClassifier(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # [CLS] 토큰에 해당하는 위치
        pooled_output = self.dropout(pooled_output)
        return self.fc(pooled_output)

In [4]:
from tqdm import tqdm
from time import time

def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    start_time = time()

    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(data_loader, leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * input_ids.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    end_time = time()
    avg_loss = total_loss / total
    acc = correct / total
    total_time = end_time - start_time
    print(f"[Test] Loss: {avg_loss:.4f}, Accuracy: {acc:.4f}, Time: {total_time:.2f} sec")
    return avg_loss, acc

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
simple_model = SimpleClassifier(vocab_size=tokenizer.vocab_size).to(device)
bert_model = BertClassifier().to(device)

criterion = nn.CrossEntropyLoss()

In [6]:
# 가중치 불러오기
simple_weight_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\nlp_weight.pt"
bert_weight_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\bert_weight.pt"

simple_weight_model = SimpleClassifier(vocab_size=tokenizer.vocab_size).to(device)
bert_weight_model = BertClassifier().to(device)
simple_weight_model.load_state_dict(torch.load(simple_weight_path, map_location=device))
bert_weight_model.load_state_dict(torch.load(bert_weight_path, map_location=device))

print("Simple Weight Model Test")
simple_weight_loss, simple_weight_acc = evaluate_model(simple_weight_model, test_loader, criterion, device)
print("-" * 10)
print("Bert Weight Model Test")
bert_weight_loss, bert_weight_acc = evaluate_model(bert_weight_model, test_loader, criterion, device)

Simple Weight Model Test


[Test] Loss: 2.2646, Accuracy: 0.4334, Time: 25.12 sec
----------
Bert Weight Model Test


[Test] Loss: 1.0195, Accuracy: 0.7199, Time: 550.29 sec


In [7]:
# 가지치기 불러오기
simple_pruned_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\nlp_pruned.pt"
bert_pruned_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\bert_pruned.pt"

simple_pruned_model = SimpleClassifier(vocab_size=tokenizer.vocab_size).to(device)
bert_pruned_model = BertClassifier().to(device)
simple_pruned_model.load_state_dict(torch.load(simple_pruned_path, map_location=device))
bert_pruned_model.load_state_dict(torch.load(bert_pruned_path, map_location=device))

print("Simple Pruned Model Test")
simple_pruned_loss, simple_pruned_acc = evaluate_model(simple_pruned_model, test_loader, criterion, device)
print("-" * 10)
print("Bert Pruned Model Test")
bert_pruned_loss, bert_pruned_acc = evaluate_model(bert_pruned_model, test_loader, criterion, device)

Simple Pruned Model Test


[Test] Loss: 2.2305, Accuracy: 0.4329, Time: 27.59 sec
----------
Bert Pruned Model Test


[Test] Loss: 1.0128, Accuracy: 0.7204, Time: 514.81 sec


In [9]:
import onnxruntime as ort
import numpy as np

def evaluate_onnx_model(onnx_path, data_loader):
    # ONNX 세션 생성
    session = ort.InferenceSession(onnx_path)

    input_name_ids = session.get_inputs()[0].name
    input_name_mask = session.get_inputs()[1].name
    output_name = session.get_outputs()[0].name

    correct = 0
    total = 0
    start_time = time()

    for input_ids, attention_mask, labels in tqdm(data_loader, leave=False):
        # numpy 변환
        input_ids_np = input_ids.cpu().numpy().astype(np.int64)
        attention_mask_np = attention_mask.cpu().numpy().astype(np.int64)
        labels_np = labels.cpu().numpy()

        # ONNX 추론
        outputs = session.run([output_name], {
            input_name_ids: input_ids_np,
            input_name_mask: attention_mask_np
        })[0]  # shape: [batch_size, num_classes]

        preds = np.argmax(outputs, axis=1)
        correct += (preds == labels_np).sum()
        total += len(labels_np)

    acc = correct / total
    total_time = time() - start_time
    print(f"[ONNX Test] Accuracy: {acc:.4f}, Time: {total_time:.2f} sec")
    return acc

simple_onnx_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\nlp_model.onnx"
bert_onnx_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\bert_model.onnx"

evaluate_onnx_model(simple_onnx_path, test_loader)
evaluate_onnx_model(bert_onnx_path, test_loader)

[ONNX Test] Accuracy: 0.4334, Time: 25.87 sec


[ONNX Test] Accuracy: 0.7199, Time: 458.61 sec


0.7198938992042441

In [10]:
import torch

# 양자화된 전체 모델 불러오기
simple_quantized_model = torch.load(
    r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\nlp_quantized.pt",
    map_location=device,
    weights_only=False
)
bert_quantized_model = torch.load(
    r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\nlp\bert_quantized.pt",
    map_location=device,
    weights_only=False
)

simple_quantized_model.eval()
bert_quantized_model.eval()

# 테스트 실행
print("Simple Quantized Model Test")
simple_loss, simple_acc = evaluate_model(simple_quantized_model, test_loader, criterion, device)
print("-" * 30)
print("Bert Quantized Model Test")
bert_loss, bert_acc = evaluate_model(bert_quantized_model, test_loader, criterion, device)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.SimpleClassifier was not an allowed global by default. Please use `torch.serialization.add_safe_globals([SimpleClassifier])` or the `torch.serialization.safe_globals([SimpleClassifier])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by d